In [3]:
from src import Store

FILE_PATH = './data/documents/2. Региональный уровень/СТП Промышленность.docx'

store = Store(FILE_PATH)
retrieve_tool = store.tool

In [4]:
from src import Model
from pydantic import Field

class ObjectModel(Model):
    """
    Описание объекта
    """
    name : str = Field(description='Наименование объекта')
    location : str = Field(description='Расположение объекта')
    params : str = Field(description='Характеристики объекта')

class SchemaModel(Model):
    """
    Результаты анализа документа схемы территориального планирования
    """
    objects : list[ObjectModel] = Field(min_length=0, description='Перечень объектов, планируемых к размещению')

In [5]:
from langgraph.graph import MessagesState
from typing import Annotated, TypedDict
import operator

class RagState(MessagesState):
    query : str
    structured_output : SchemaModel

In [ ]:
from src import llm
from langchain.messages import SystemMessage, HumanMessage

router_prompt = """
ПРАВИЛА РАБОТЫ:
- УДОСТОВЕРЬСЯ, что информации в диалоге достаточно для ответа согласно схеме. Иначе вызови поиск.
- ПРОВЕРЬ, что найденная информация полная. Вызови поиск для уточнения. 
- НЕ ПРИДУМЫВАЙ информацию и не пользуйся общими знаниями.
---
ЗАПРОС:
{query}
"""

def router_node(state : RagState):
    """
    Проверяет достаточность данных для формирования ответа
    """
    response = llm.with_structured_output(
        SchemaModel,
        strict=True,
        tools=[retrieve_tool]
    ).invoke([
        SystemMessage(router_prompt.format(
            query=state['query']
        )),
        *state['messages']
    ])
    return {
        'messages': [response]
    }

In [10]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

graph = StateGraph(RagState)
graph.add_node('router', router_node)
graph.add_node('tool', ToolNode([retrieve_tool]))

graph.add_edge(START, 'router')

graph.add_conditional_edges(
    'router',
    tools_condition,
    {
        'tools': 'tool',
        END: END,
    }
)
graph.add_edge('tool', 'router')

app = graph.compile()

In [11]:
rag_state = {
    'query': 'г. Гатчина, Гатчинский муниципальный округ (район)',
    'messages': [],
    'router_approve': False,
    'current_iteration' : 0,
    'max_iteration': 3
}

res = app.invoke(rag_state, debug=True)

[values] {'messages': [], 'query': 'г. Гатчина, Гатчинский муниципальный округ (район)'}
None


NotImplementedError: Unsupported message type: <class 'NoneType'>
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/MESSAGE_COERCION_FAILURE 

In [ ]:
res